# GNM-Contact Learner: 2000 PDB Scale-Up Training

4-phase pipeline:
1. **PDB Curation** — RCSB API ile 2000 diverse protein
2. **Chunked Inference** — 10'arlık batch ile OpenFold3 pair repr extraction
3. **Shard Packing** — .pt → .npz (50/shard)
4. **Training** — Focal loss + OneCycleLR + gradient accumulation

Her cell resume-safe. Colab disconnect olursa kaldığın yerden devam eder.

## 1. Environment Setup

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 1: Install Dependencies
# ══════════════════════════════════════════════════════════════════
!pip install -q biopython requests wandb

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 2: Clone / Update Repository
# ══════════════════════════════════════════════════════════════════
import os
from pathlib import Path

REPO_DIR = Path('/content/ANM-openfold3')

if REPO_DIR.exists():
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/beratkaskaloglu/ANM-openfold3.git {REPO_DIR}

os.chdir(str(REPO_DIR))
print(f"Working dir: {os.getcwd()}")
!git log --oneline -5

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 3: Setup OpenFold3 (clone + install)
# ══════════════════════════════════════════════════════════════════
OF3_DIR = Path('/content/ANM-openfold3/openfold3-repo')

if not OF3_DIR.exists():
    !git clone https://github.com/aqlaboratory/openfold-3.git {OF3_DIR}
    !pip install -e {OF3_DIR}
else:
    print("OpenFold3 already installed")

import sys
sys.path.insert(0, str(OF3_DIR))
sys.path.insert(0, str(REPO_DIR))

## 2. Phase 1 — PDB Curation (2000 proteins)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 4: Fetch 2000 diverse PDBs from RCSB
#
#  Filters: X-ray, ≤2.5Å, 30-500 aa, ≤30% seq identity
#  Resume-safe: skips if data/pdb_2000.json already exists
# ══════════════════════════════════════════════════════════════════
import json

PDB_LIST = Path('data/pdb_2000.json')

if PDB_LIST.exists():
    proteins = json.loads(PDB_LIST.read_text())
    print(f"PDB list already exists: {len(proteins)} proteins")
else:
    !python scripts/fetch_pdb_list.py --target 2000
    proteins = json.loads(PDB_LIST.read_text())
    print(f"Fetched: {len(proteins)} proteins")

# Stats
lengths = [p['length'] for p in proteins]
print(f"Length: min={min(lengths)}, max={max(lengths)}, "
      f"mean={sum(lengths)/len(lengths):.0f}, "
      f"median={sorted(lengths)[len(lengths)//2]}")

## 3. Phase 2+3 — Chunked Inference + Inline Shard Packing

10 protein / inference chunk, 50 protein / .npz shard.

Her 50 protein'den sonra:
1. .pt → .npz shard'a pakle
2. .pt dosyalarını sil (disk max ~1-2 GB)

**~100s / protein → ~55 saat total.**
Resume-safe: tamamlanan shard'lar skip edilir.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 5: Directories
# ══════════════════════════════════════════════════════════════════
PAIR_REPR_DIR = Path('/content/pair_reprs')
COORDS_DIR    = Path('/content/coords')
OF3_OUTPUT    = Path('/content/of3_output')
QUERY_DIR     = Path('/content/queries')
PROGRESS_DIR  = Path('data/progress')
SHARD_DIR     = Path('data/shards')
CKPT_DIR      = Path('checkpoints')

for d in [PAIR_REPR_DIR, COORDS_DIR, OF3_OUTPUT, QUERY_DIR,
          PROGRESS_DIR, SHARD_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
    print(f"  {d}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 6: Run Inference + Pack Shards (combined)
#
#  Her 50 protein → 1 .npz shard → .pt siliniyor
#  Disk kullanimi max ~1-2 GB
#  Disconnect olursa: bu cell'i tekrar calistir, devam eder
# ══════════════════════════════════════════════════════════════════
START = 0
END   = 2000   # -1 = all

!PYTHONPATH=/content/ANM-openfold3/openfold3-repo:$PYTHONPATH python scripts/extract_pairs.py \
    --pdb-list data/pdb_2000.json \
    --shard-size 50 \
    --inference-chunk-size 10 \
    --start {START} \
    --end {END} \
    --pair-repr-dir {PAIR_REPR_DIR} \
    --coords-dir {COORDS_DIR} \
    --output-dir {OF3_OUTPUT} \
    --query-dir {QUERY_DIR} \
    --shard-dir {SHARD_DIR} \
    --progress-dir {PROGRESS_DIR}

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 7: Progress Check
# ══════════════════════════════════════════════════════════════════
import json

ok_files = sorted(PROGRESS_DIR.glob('shard_*.ok'))
total_ok = 0
total_fail = 0
all_failed = []

for f in ok_files:
    info = json.loads(f.read_text())
    total_ok += info['success']
    total_fail += len(info.get('failed', []))
    all_failed.extend(info.get('failed', []))

n_shards = len(list(SHARD_DIR.glob('shard_*.npz')))
total_size = sum(f.stat().st_size for f in SHARD_DIR.glob('shard_*.npz')) / 1e6

print(f'Shards: {n_shards} files ({total_size:.0f} MB)')
print(f'Success: {total_ok}, Failed: {total_fail}')
if all_failed:
    print(f'Failed: {all_failed[:20]}{"..." if len(all_failed) > 20 else ""}')

## 4. Phase 4 — Training

Shard packing artik inference ile birlikte yapiliyor.
Direkt training'e gecebilirsin.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 8: Verify Shards
# ══════════════════════════════════════════════════════════════════
import numpy as np

shard_files = sorted(SHARD_DIR.glob('shard_*.npz'))
total_proteins = 0
for sf in shard_files:
    data = np.load(sf, allow_pickle=True)
    n = len(data['pdb_ids'])
    total_proteins += n

print(f'Total shards: {len(shard_files)}')
print(f'Total proteins: {total_proteins}')
print(f'Ready for training!')

## 5. Training

**Hyperparameters:**
- Focal Loss (gamma=2.0, alpha=0.75)
- OneCycleLR (warmup 5% + cosine decay)
- Gradient accumulation (4 steps)
- r_cut=8.0 A, tau=1.0
- bottleneck_dim=64
- Early stopping (patience=50)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 9: Kernel restart gerekirse — sadece bu cell'i çalıştır
#  GPU belleği temizlemek için:
# ══════════════════════════════════════════════════════════════════
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {free/1e9:.1f} / {total/1e9:.1f} GB free")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 10: Train!
#
#  Resume-safe: --resume checkpoints/latest.pt
#  Colab disconnect olursa bu cell'i tekrar çalıştır,
#  kaldığı epoch'tan devam eder.
# ══════════════════════════════════════════════════════════════════
EPOCHS = 500
LR = 3e-4
BOTTLENECK_DIM = 64
R_CUT = 8.0
TAU = 1.0
PATIENCE = 50

# İlk çalıştırma: resume yok
# Disconnect sonrası: RESUME = 'checkpoints/latest.pt'
RESUME = ''  # veya 'checkpoints/latest.pt'

resume_flag = f'--resume {RESUME}' if RESUME else ''

!PYTHONPATH=/content/ANM-openfold3/openfold3-repo:$PYTHONPATH python scripts/train_large.py \
    --epochs {EPOCHS} \
    --lr {LR} \
    --bottleneck-dim {BOTTLENECK_DIM} \
    --r-cut {R_CUT} \
    --tau {TAU} \
    --patience {PATIENCE} \
    --shard-dir {SHARD_DIR} \
    --ckpt-dir {CKPT_DIR} \
    --log-every 5 \
    --save-every 50 \
    {resume_flag}

## 6. Results & Visualization

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 11: Training Curves
# ══════════════════════════════════════════════════════════════════
import json
import matplotlib.pyplot as plt

history = json.loads((CKPT_DIR / 'history.json').read_text())

epochs = [h['epoch'] for h in history]
train_loss = [h['train_loss'] for h in history]
val_loss = [h['val_loss'] for h in history]
val_acc = [h['val_adj_acc'] for h in history]
val_bf = [h['val_bf_pearson'] for h in history]
lrs = [h['lr'] for h in history]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0,0].plot(epochs, train_loss, label='Train', alpha=0.7)
axes[0,0].plot(epochs, val_loss, label='Val', alpha=0.7)
axes[0,0].set_xlabel('Epoch')
axes[0,0].set_ylabel('Total Loss')
axes[0,0].set_title('Loss')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# Adjacency Accuracy
axes[0,1].plot(epochs, val_acc, color='green')
axes[0,1].set_xlabel('Epoch')
axes[0,1].set_ylabel('Accuracy')
axes[0,1].set_title('Val Adjacency Accuracy')
axes[0,1].set_ylim(0, 1)
axes[0,1].grid(True, alpha=0.3)

# B-factor correlation
axes[1,0].plot(epochs, val_bf, color='orange')
axes[1,0].set_xlabel('Epoch')
axes[1,0].set_ylabel('Pearson r')
axes[1,0].set_title('Val B-factor Correlation')
axes[1,0].set_ylim(-0.2, 1)
axes[1,0].grid(True, alpha=0.3)

# Learning rate
axes[1,1].plot(epochs, lrs, color='red')
axes[1,1].set_xlabel('Epoch')
axes[1,1].set_ylabel('LR')
axes[1,1].set_title('Learning Rate Schedule')
axes[1,1].set_yscale('log')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(CKPT_DIR / 'training_curves.png'), dpi=150)
plt.show()
print(f"Saved: {CKPT_DIR / 'training_curves.png'}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 12: Test Results
# ══════════════════════════════════════════════════════════════════
test_results = json.loads((CKPT_DIR / 'test_results.json').read_text())

print("=" * 50)
print("TEST SET RESULTS")
print("=" * 50)
print(f"Best epoch:      {test_results['best_epoch']}")
print(f"Best val loss:   {test_results['best_val_loss']:.4f}")
print(f"Total epochs:    {test_results['total_epochs']}")
print(f"Training time:   {test_results['total_time_min']:.1f} min")
print()
metrics = test_results['test_metrics']
print(f"Test loss:       {metrics['loss']:.4f}")
print(f"Adj accuracy:    {metrics.get('adj_acc', 0):.4f}")
print(f"B-factor r:      {metrics.get('bf_pearson', 0):.4f}")
print(f"L_contact:       {metrics.get('L_contact', 0):.4f}")
print(f"L_gnm:           {metrics.get('L_gnm', 0):.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 13: Contact Map Visualization (Test Set Samples)
# ══════════════════════════════════════════════════════════════════
import sys, numpy as np
sys.path.insert(0, '/content/ANM-openfold3')

from src.contact_head import ContactProjectionHead
from src.data import ShardedPairReprDataset
from src.kirchhoff import soft_kirchhoff, gnm_decompose
from src.ground_truth import compute_gt_probability_matrix
from torch.utils.data import Subset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load best model
ckpt = torch.load(CKPT_DIR / 'best_model.pt', map_location=DEVICE, weights_only=False)
head = ContactProjectionHead(
    c_z=ckpt['c_z'],
    bottleneck_dim=ckpt['bottleneck_dim']
).to(DEVICE)
head.load_state_dict(ckpt['model_state_dict'])
head.eval()
print(f"Loaded best model (epoch {ckpt['epoch']+1}, val_loss={ckpt['val_loss']:.4f})")

# Load dataset for test samples
shard_paths = sorted(SHARD_DIR.glob('shard_*.npz'))
dataset = ShardedPairReprDataset(shard_paths, r_cut=R_CUT, tau=TAU)

# Show 4 random samples
n_show = min(4, len(dataset))
rng = np.random.RandomState(42)
show_idx = rng.choice(len(dataset), n_show, replace=False)

fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))
if n_show == 1:
    axes = axes[None, :]

with torch.no_grad():
    for row, idx in enumerate(show_idx):
        sample = dataset[idx]
        z = sample['pair_repr'].to(DEVICE)
        c_gt = sample['c_gt'].squeeze().cpu().numpy()
        c_pred = head(z).squeeze().cpu().numpy()
        pdb_id = sample['pdb_id']

        axes[row, 0].imshow(c_gt, cmap='hot', vmin=0, vmax=1)
        axes[row, 0].set_title(f'{pdb_id} — Ground Truth')

        axes[row, 1].imshow(c_pred, cmap='hot', vmin=0, vmax=1)
        axes[row, 1].set_title(f'{pdb_id} — Predicted')

        diff = np.abs(c_pred - c_gt)
        axes[row, 2].imshow(diff, cmap='Blues', vmin=0, vmax=0.5)
        axes[row, 2].set_title(f'{pdb_id} — |Diff|')

plt.tight_layout()
plt.savefig(str(CKPT_DIR / 'contact_maps_2000.png'), dpi=150)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 14: B-factor Profile Comparison
# ══════════════════════════════════════════════════════════════════
N_MODES = 20

def pearson_corr(x, y):
    xc = x - x.mean()
    yc = y - y.mean()
    return ((xc * yc).sum() / (xc.norm() * yc.norm()).clamp(min=1e-8)).item()

n_show = min(4, len(dataset))
fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 4))
if n_show == 1:
    axes = [axes]

with torch.no_grad():
    for i, idx in enumerate(show_idx):
        sample = dataset[idx]
        z = sample['pair_repr'].to(DEVICE)
        c_gt = sample['c_gt'].squeeze().to(DEVICE)
        c_pred = head(z).squeeze()
        pdb_id = sample['pdb_id']

        _, _, bf_gt = gnm_decompose(soft_kirchhoff(c_gt), N_MODES)
        _, _, bf_pred = gnm_decompose(soft_kirchhoff(c_pred), N_MODES)

        bf_gt_n = (bf_gt / bf_gt.max()).cpu().numpy()
        bf_pred_n = (bf_pred / bf_pred.max()).cpu().numpy()
        r = pearson_corr(bf_pred.cpu(), bf_gt.cpu())

        axes[i].plot(bf_gt_n, label='GT', linewidth=2)
        axes[i].plot(bf_pred_n, label='Pred', linewidth=2, linestyle='--')
        axes[i].set_title(f'{pdb_id}  (r={r:.3f})')
        axes[i].set_xlabel('Residue')
        axes[i].set_ylabel('Normalized B-factor')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(CKPT_DIR / 'bfactor_profiles_2000.png'), dpi=150)
plt.show()

## 7. Export & Download

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 15: Download Results
# ══════════════════════════════════════════════════════════════════
import shutil

# Pack checkpoints + results into zip
zip_path = '/content/gnm_contact_2000_results'
shutil.make_archive(
    zip_path, 'zip',
    root_dir=str(CKPT_DIR.parent),
    base_dir=CKPT_DIR.name
)
print(f"Results packed: {zip_path}.zip")
print(f"Size: {Path(zip_path + '.zip').stat().st_size / 1e6:.1f} MB")

# If using Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy(f'{zip_path}.zip', '/content/drive/MyDrive/')

# Or download directly:
try:
    from google.colab import files
    files.download(f'{zip_path}.zip')
except ImportError:
    print('Not in Colab — use VSCode file explorer to download')